A proof of concept here is the brochure pipeline pointed at a different document. One model call can gather or clean the source. A second call writes the finished piece from that source only.

## Blueprint

**1. Name the document and the reader.**  
Decide the exact output and who it is for. One document type, one audience. “A one-page product brief for a new sales hire” is a task. “Help the sales team” is not.

**2. Name the source.**  
The model should write from material you supply, not from memory. That source might be a webpage, a spec, a job description, meeting notes, or a customer record. If the source is long, plan to keep only the useful part.

**3. First call: select.**  
Ask the model to pull out only what the document needs: relevant pages, relevant fields, or relevant facts. The result of this call is a short fact list, not the finished document.

**4. Second call: write.**  
Give the model a role, the audience, the format, and the tone. Put the fact list from step 3 in the user prompt. Tell it to use only those facts and to skip anything the source does not say.

**5. Check it by hand.**  
Read one result and ask: did it invent anything, did it miss the main point, and would the real reader use it? Then try a second source of the same kind. Two good runs is enough for a proof of concept.

## Two tasks that fit this well

**Task 1 - New-hire product brief.**  
Source: a product page plus a short internal spec or FAQ. First call extracts what the product does, who it is for, pricing, and limits. Second call writes a one-page brief a new salesperson can read before a call.

**Task 2- Customer follow-up email.**  
Source: notes from one sales or support call, plus the company’s usual offer. First call extracts the customer’s problem, what was agreed, and any open question. Second call writes a short email in your company’s tone, with a clear next step, and nothing that was not in the notes.

## Task 1: New-hire product brief  
**Source:** a product page plus a short internal spec or FAQ. First call extracts what the product does, who it is for, pricing, and limits. Second call writes a one-page brief a new salesperson can read before a call.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents  
from openai import OpenAI


In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
content_system_prompt = """
You are provided with the content of a product page found on a webpage.
You are able to extracts what the product does, who it is for, pricing, what the product does not do and limits.

You should respond in JSON as in this example:

{
    "product": [
        {"type": "phone"},
        {"brand": "Nokia"},
        {"price": "$1000"},
        {"who-is-it-for": "for smart users"},
        {"does": "Smart phone features"},
        {"does-not-do": "It cannot fly"}
    ]
}
"""

In [4]:
def get_user_prompt(product_details):
    content_user_prompt = f"""

Please extract the product spoecifications from the product page and
respond details in JSON format.
Here is the content of the product page:
"""
    content_user_prompt += "\n".join(product_details)
    return content_user_prompt

In [5]:
def get_product_specs(product_source):
    product_page_contents = fetch_website_contents(product_source)
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": content_system_prompt},
            {"role": "user", "content": get_user_prompt(product_page_contents)}
            ],
        )
    result = response.choices[0].message.content
    return result
        # display(Markdown(result))

In [10]:
product_brief_system_prompt = """
You are an assistant that analyzes the specifications of a product and creates a one-page product brief such that a reader 
who is a salesperson in their first week, reading it the morning before a customer call should be able to say what the 
product does, who it is for, what it costs, and what it does not do, without guessing. Also, mentioning the price is one of the most important factor, do not miss this.
Out of scope for this document: a customer-facing brochure, a technical manual, and an email.
"""

In [11]:
def get_product_brief_user_prompt(specs):
    user_prompt = f"""
Here are the specifications of this product in JSON format;
use this information to build a one-page brief of the product for the salesperson in markdown without code blocks.\n\n
"""
    user_prompt += specs
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [12]:
def create_product_brief(product_source):
    specs = get_product_specs(product_source)
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": product_brief_system_prompt},
            {"role": "user", "content": get_product_brief_user_prompt(specs)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [13]:
product_source='https://www.apple.com/in/shop/buy-iphone/iphone-18-pro'
create_product_brief(product_source)

# Product Brief: Apple iPhone 18 Pro and iPhone 18 Pro Max

---

## What is it?
The Apple iPhone 18 Pro and iPhone 18 Pro Max are the latest high-end smartphones from Apple, available in multiple storage capacities and colors.

---

## Who is it for?
This product is designed for customers seeking premium Apple smartphones with large storage options to accommodate extensive apps, media, and data.

---

## Available Storage Options & Pricing
The iPhone 18 Pro series offers four storage variants:

| Storage | Price (₹)           | Monthly EMI (₹) with Instant Cashback & No-Cost EMI |
|---------|--------------------|----------------------------------------------------|
| 256 GB  | ₹179,900           | ₹28,817                                            |
| 512 GB  | ₹204,900           | ₹32,983                                            |
| 1 TB    | ₹254,900           | ₹41,317                                            |
| 2 TB    | ₹329,900           | ₹53,817                                            |

*Prices are consistent across all colors.*

---

## Colors Available
- Burgundy
- Glacier
- Silver
- Black

---

## Key Features
- Latest Apple iPhone 18 Pro and iPhone 18 Pro Max models
- Multiple large storage capacities (256 GB to 2 TB)
- Four premium color options

---

## What this product does NOT include
- No detailed information on specific technical features, specs, camera, battery life, or software capabilities is provided at this time.

---

## Important Notes & Limitations
- Pricing and monthly installment offers depend on the availability of instant cashback and no-cost EMI options.
- Only the specified colors and storage configurations are available.
- Sales personnel should verify the availability of these offers before finalizing the sale.

---

This brief should equip you to clearly communicate the essential details of the Apple iPhone 18 Pro lineup, including pricing, options, and target customer profile.

In [14]:
product_source='https://www.amazon.in/gp/product/B0HH9GJQTH/ref=ox_sc_act_title_3?smid=A18F28GKTXT7LZ&th=1'
create_product_brief(product_source)

# Product Brief: KENT Grand RO Water Purifier

**What it is:**  
The KENT Grand RO is an advanced water purifier designed to provide safe and clean drinking water using multiple purification technologies.

**Who it is for:**  
This purifier is ideal for households or small offices dealing with water from diverse sources such as borewell, tanker, and municipal supplies. It's suitable for customers looking for reliable purification that handles high TDS and ensures water safety with multiple filtration stages.

**Key Features & Technology:**  
- **Advanced Reverse Osmosis (RO) Technology:** Removes dissolved impurities effectively.  
- **Ultrafiltration (UF):** Further filters out bacteria and cysts.  
- **Total Dissolved Solids (TDS) Control:** Ensures the water retains essential minerals for taste and health.  
- **UV Purification with LED Indicators:** Kills viruses and bacteria and clearly indicates purifier status.  
- **7.5 Liters Capacity:** Suitable for daily family water needs.  
- **Flow Rate:** Purifies water at 20 Liters Per Hour (LPH).  
- **Multi-Source Compatibility:** Works efficiently with borewell, tanker, and municipal water sources.  
- **Color:** Sleek Black finish that fits modern kitchen aesthetics.  
- **Service Network:** Supported by the largest service network ensuring easy maintenance and support.  
- **Availability:** Available for purchase on Amazon.in under Home & Kitchen category.

**Pricing:**  
The exact price is not specified in the provided information. Please check Amazon.in for current pricing details.

**What it does NOT do:**  
- Does not provide heating or refrigeration functions.  
- Not designed for industrial or very large-scale water purification needs.  
- Lacks smart connectivity or Wi-Fi features for remote monitoring.  
- Does not have a mineral cartridge replacement alert system explicitly mentioned.

---

**Summary for the Salesperson:**  
The KENT Grand RO is a premium water purifier featuring RO, UF, UV, and TDS control technology to treat water from multiple sources, ensuring safe drinking water with essential mineral retention. It offers a good capacity of 7.5 liters and a high flow rate of 20 LPH, making it suitable for families using borewell, tanker, or municipal water. Durable and user-friendly with LED indicators and supported by a strong service network. Pricing should be verified on Amazon.in before discussing with customers.

## Task 2: Customer follow-up email 
**Source:** notes from one sales or support call, plus the company’s usual offer. First call extracts the customer’s problem, what was agreed, and any open question. Second call writes a short email in your company’s tone, with a clear next step, and nothing that was not in the notes.

In [ ]:
notes_source = "notes-sales-call.txt"

with open(notes_source, "r", encoding="utf-8") as file:
    sales_notes = file.read()

In [ ]:
# STEP 1 
# =========
notes_system_prompt="""
You are provided with notes from one sales or support call, plus the company's usual offer.
You are able to extracts the customers' proble, what was agreed and any open question.
"""

In [23]:
notes_user_prompt="""
Here is the notes log of a recent sales call, get all relevant details from this call notes:
"""

notes_user_prompt+="\n\n"+ sales_notes

In [30]:
response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": notes_system_prompt},
        {"role": "user", "content": notes_user_prompt}
        ],
    )
result = response.choices[0].message.content
print(result)

Here is the extracted relevant information from the sales call notes:

---

### Customer's Problem / Needs
- Current support management uses shared Gmail inbox and spreadsheets.
- Multiple agents handling same inbox results in missed or duplicated requests.
- Needs centralized platform for:
  - Tracking support tickets
  - Monitoring agent performance
  - Ticket assignment automation
  - Automated email notifications
  - Response-time tracking
  - Basic and advanced reporting
- Support volume: approximately 800–1,000 requests per month.
- Team size: 12 support agents.
- Considering migration to dedicated support platform within 30–45 days.
- Budget: ₹15,000–₹20,000 per month.

### What Was Agreed / Discussed
- Demonstration of ticket management dashboard and automated assignment feature was given.
- Customer liked automated assignment and reporting features.
- Customer asked about the possibility of importing historical Gmail conversations into the support platform.
- Agreed actions:
 

In [31]:
# STEP 2 
# =========
email_system_prompt="""
You are a helpful assistant with expert professional business writing skills keeping the tone as per company's policy.
"""

In [32]:
email_user_prompt="""
Here is the relevant details extract from a recent sales call.
Writes a short email in the company’s tone, with a clear next step, and nothing that was not in the notes.
"""
email_user_prompt+="\n\n".join(result)

In [34]:
response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": email_system_prompt},
        {"role": "user", "content": email_user_prompt}
        ],
    )
result = response.choices[0].message.content
display(Markdown(result))

Subject: Next Steps: Pricing Proposal and Gmail Import Feasibility

Dear [Customer Name],

Thank you for the time spent reviewing our ticket management and automated assignment features. We are pleased to know that these functionalities meet your expectations.

As discussed, the next steps are:
- We will share a tailored pricing proposal for your team of 12 agents.
- We will confirm the technical feasibility and details regarding the import of your historical Gmail conversation data into our platform.

We aim to provide this information for your review by 20 November 2026 to assist in your decision-making process.

Please let us know if you have any additional questions or require further clarification.

Best regards,  
[Your Name]  
[Your Position]  
[Company Name]